# Convert Topologic Graph into a Neo4j Graph

This notebook demonstrates how to:

1. Build a 3D model of a two-storey house as a CellComplex using topologic_fast
2. Create a dual graph representing room connectivity
3. Visualize the graph with Plotly
4. Prepare the graph for Neo4j export

**Note**: The actual Neo4j export functionality (`Neo4j.ByGraph`) is not yet implemented in topologic_fast. This notebook focuses on graph preparation and shows the structure needed for Neo4j integration.

## Requirements

- topologic_fast
- plotly
- neo4j (optional, for actual database export)

In [ ]:
import topologic_fast as tf
import plotly.graph_objects as go
import uuid
import json

## Check Version

In [ ]:
print(f"topologic_fast version: {tf.__version__}")

## Build the House

We'll create a two-storey house with multiple rooms including:
- Ground floor: Living room, Kitchen, Corridor, Bedroom, Bathroom, Home Office, Stairwell
- First floor: Same layout as ground floor

In [ ]:
# Room metadata storage
room_metadata = []

def create_room(x, y, z, width, length, height, name, room_type="Space"):
    """
    Create a room cell with metadata.
    """
    room = tf.Cell.Box(x, y, z, width, length, height)
    room_id = str(uuid.uuid4())
    
    # Store metadata separately since Dictionary integration is limited
    metadata = {
        "id": room_id,
        "name": name,
        "type": room_type,
        "volume": room.Volume(),
        "centroid": room.CenterOfMass().Coordinates()
    }
    room_metadata.append(metadata)
    
    return room, metadata

In [ ]:
# Ground Floor (z = 0)
floor_height = 3.0

# Living Room
living_room_gf, lr_meta = create_room(0, 0, 0, 3, 5, floor_height, "Living Room GF")

# Kitchen
kitchen_gf, kt_meta = create_room(0, 5, 0, 3, 4, floor_height, "Kitchen GF")

# Corridor
corridor_gf, cr_meta = create_room(3, 2, 0, 2, 6, floor_height, "Corridor GF")

# Bedroom
bedroom_gf, br_meta = create_room(5, 0, 0, 4, 4, floor_height, "Bedroom GF")

# Bathroom
bathroom_gf, bt_meta = create_room(5, 4, 0, 3, 2, floor_height, "Bathroom GF")

# Home Office
home_office_gf, ho_meta = create_room(5, 6, 0, 4, 3, floor_height, "Home Office GF")

# Stairwell (spans both floors)
stairwell, sw_meta = create_room(3.2, 8, 0, 1.6, 3.2, floor_height * 2, "Stairwell")

ground_floor_rooms = [living_room_gf, kitchen_gf, corridor_gf, bedroom_gf, bathroom_gf, home_office_gf]

print(f"Ground floor rooms created: {len(ground_floor_rooms)}")

In [ ]:
# First Floor (z = 3)
z_offset = floor_height

# Living Room
living_room_ff, _ = create_room(0, 0, z_offset, 3, 5, floor_height, "Living Room 1F")

# Kitchen
kitchen_ff, _ = create_room(0, 5, z_offset, 3, 4, floor_height, "Kitchen 1F")

# Corridor
corridor_ff, _ = create_room(3, 2, z_offset, 2, 6, floor_height, "Corridor 1F")

# Bedroom
bedroom_ff, _ = create_room(5, 0, z_offset, 4, 4, floor_height, "Bedroom 1F")

# Bathroom
bathroom_ff, _ = create_room(5, 4, z_offset, 3, 2, floor_height, "Bathroom 1F")

# Home Office
home_office_ff, _ = create_room(5, 6, z_offset, 4, 3, floor_height, "Home Office 1F")

first_floor_rooms = [living_room_ff, kitchen_ff, corridor_ff, bedroom_ff, bathroom_ff, home_office_ff]

print(f"First floor rooms created: {len(first_floor_rooms)}")

In [ ]:
# Combine all rooms into a CellComplex
all_rooms = ground_floor_rooms + first_floor_rooms + [stairwell]
house = tf.CellComplex.ByCells(all_rooms)

print(f"House CellComplex created:")
print(f"  Total Cells: {house.NumCells()}")
print(f"  Total Volume: {house.Volume():.1f} m^3")
print(f"  Total Surface Area: {house.Area():.1f} m^2")

## Room Summary

In [ ]:
print("\nRoom Details:")
print("=" * 60)
for meta in room_metadata:
    print(f"  {meta['name']:20s} | Volume: {meta['volume']:6.1f} m^3 | ID: {meta['id'][:8]}...")

## Create Dual Graph

The dual graph represents room connectivity where:
- **Vertices** = Room centroids
- **Edges** = Shared walls/faces between rooms

In [ ]:
# Create the dual graph from the CellComplex
house_graph = tf.Graph.ByTopology(house)

print(f"\nHouse Graph:")
print(f"  Vertices (rooms): {house_graph.Order()}")
print(f"  Edges (connections): {house_graph.Size()}")
print(f"  Density: {house_graph.Density():.3f}")
print(f"  Diameter: {house_graph.Diameter()} steps")
print(f"  Is Bipartite: {house_graph.IsBipartite()}")

## Graph Analysis

In [ ]:
# Get graph vertices and analyze connectivity
graph_vertices = house_graph.Vertices()

print("\nRoom Connectivity:")
print("=" * 60)

connectivity_data = []
for i, vertex in enumerate(graph_vertices):
    adjacent = house_graph.AdjacentVertices(vertex)
    degree = house_graph.VertexDegree(vertex)
    coords = vertex.Coordinates()
    
    # Find matching room metadata by centroid proximity
    room_name = f"Room {i}"
    for meta in room_metadata:
        mc = meta['centroid']
        dist = ((coords[0] - mc[0])**2 + (coords[1] - mc[1])**2 + (coords[2] - mc[2])**2)**0.5
        if dist < 0.5:
            room_name = meta['name']
            break
    
    connectivity_data.append({
        'name': room_name,
        'degree': degree,
        'coords': coords
    })
    
    print(f"  {room_name:20s} | {degree} connections | at ({coords[0]:.1f}, {coords[1]:.1f}, {coords[2]:.1f})")

## Visualize the House Graph

In [ ]:
def visualize_house_with_graph(cellcomplex, graph, room_metadata):
    """
    Create a 3D visualization of the house with the connectivity graph.
    """
    fig = go.Figure()
    
    # Color map for room types
    room_colors = {
        'Living Room': 'lightgreen',
        'Kitchen': 'peachpuff',
        'Bedroom': 'lightblue',
        'Bathroom': 'lavender',
        'Corridor': 'lightgray',
        'Home Office': 'lightyellow',
        'Stairwell': 'orange'
    }
    
    # Draw cells (rooms)
    cells = cellcomplex.Cells()
    for i, cell in enumerate(cells):
        faces = cell.Faces()
        
        # Determine color based on room name
        color = 'lightgray'
        centroid = cell.CenterOfMass().Coordinates()
        for meta in room_metadata:
            mc = meta['centroid']
            dist = ((centroid[0] - mc[0])**2 + (centroid[1] - mc[1])**2 + (centroid[2] - mc[2])**2)**0.5
            if dist < 0.5:
                for rtype, rcolor in room_colors.items():
                    if rtype in meta['name']:
                        color = rcolor
                        break
                break
        
        for face in faces:
            vertices = face.Vertices()
            coords = [v.Coordinates() for v in vertices]
            
            if len(coords) >= 3:
                x = [c[0] for c in coords]
                y = [c[1] for c in coords]
                z = [c[2] for c in coords]
                
                fig.add_trace(go.Mesh3d(
                    x=x, y=y, z=z,
                    color=color,
                    opacity=0.3,
                    alphahull=0,
                    showlegend=False
                ))
                
                # Add edges
                for k in range(len(coords)):
                    p1 = coords[k]
                    p2 = coords[(k + 1) % len(coords)]
                    fig.add_trace(go.Scatter3d(
                        x=[p1[0], p2[0]], y=[p1[1], p2[1]], z=[p1[2], p2[2]],
                        mode='lines',
                        line=dict(color='black', width=1),
                        showlegend=False,
                        hoverinfo='skip'
                    ))
    
    # Draw graph edges
    graph_edges = graph.Edges()
    for edge in graph_edges:
        edge_verts = edge.Vertices()
        if len(edge_verts) == 2:
            p1 = edge_verts[0].Coordinates()
            p2 = edge_verts[1].Coordinates()
            fig.add_trace(go.Scatter3d(
                x=[p1[0], p2[0]],
                y=[p1[1], p2[1]],
                z=[p1[2], p2[2]],
                mode='lines',
                line=dict(color='red', width=5),
                showlegend=False,
                hoverinfo='skip'
            ))
    
    # Draw graph vertices with labels
    graph_vertices = graph.Vertices()
    vertex_labels = []
    for vertex in graph_vertices:
        coords = vertex.Coordinates()
        label = "Room"
        for meta in room_metadata:
            mc = meta['centroid']
            dist = ((coords[0] - mc[0])**2 + (coords[1] - mc[1])**2 + (coords[2] - mc[2])**2)**0.5
            if dist < 0.5:
                label = meta['name']
                break
        vertex_labels.append(label)
    
    vertex_coords = [v.Coordinates() for v in graph_vertices]
    x = [c[0] for c in vertex_coords]
    y = [c[1] for c in vertex_coords]
    z = [c[2] for c in vertex_coords]
    
    fig.add_trace(go.Scatter3d(
        x=x, y=y, z=z,
        mode='markers+text',
        marker=dict(size=8, color='red'),
        text=vertex_labels,
        textposition='top center',
        textfont=dict(size=9),
        name='Graph Nodes',
        hovertext=vertex_labels,
        hoverinfo='text'
    ))
    
    fig.update_layout(
        title='Two-Storey House with Room Connectivity Graph',
        scene=dict(
            aspectmode='data',
            xaxis_title='X (m)',
            yaxis_title='Y (m)',
            zaxis_title='Z (m)',
            camera=dict(eye=dict(x=1.5, y=-1.5, z=1.0))
        ),
        width=1000,
        height=800
    )
    
    return fig

fig = visualize_house_with_graph(house, house_graph, room_metadata)
fig.show()

## Prepare Data for Neo4j Export

The following section shows how to structure the graph data for Neo4j import. While the actual `Neo4j.ByGraph()` function is not yet implemented in topologic_fast, we can prepare the data in a format suitable for neo4j-python-driver.

In [ ]:
# Prepare node data for Neo4j
neo4j_nodes = []

graph_vertices = house_graph.Vertices()
for i, vertex in enumerate(graph_vertices):
    coords = vertex.Coordinates()
    
    # Find matching room metadata
    node_data = {
        'label': f'Room_{i}',
        'name': f'Room {i}',
        'type': 'Space',
        'x': coords[0],
        'y': coords[1],
        'z': coords[2],
        'degree': house_graph.VertexDegree(vertex)
    }
    
    for meta in room_metadata:
        mc = meta['centroid']
        dist = ((coords[0] - mc[0])**2 + (coords[1] - mc[1])**2 + (coords[2] - mc[2])**2)**0.5
        if dist < 0.5:
            node_data.update({
                'label': meta['id'],
                'name': meta['name'],
                'volume': meta['volume']
            })
            break
    
    neo4j_nodes.append(node_data)

print("Neo4j Node Data:")
print(json.dumps(neo4j_nodes[:3], indent=2))
print(f"... and {len(neo4j_nodes) - 3} more nodes")

In [ ]:
# Prepare edge data for Neo4j
neo4j_edges = []

graph_edges = house_graph.Edges()
for edge in graph_edges:
    edge_verts = edge.Vertices()
    if len(edge_verts) == 2:
        p1 = edge_verts[0].Coordinates()
        p2 = edge_verts[1].Coordinates()
        
        # Find node indices
        idx1 = idx2 = -1
        for i, vertex in enumerate(graph_vertices):
            vc = vertex.Coordinates()
            if abs(vc[0] - p1[0]) < 0.1 and abs(vc[1] - p1[1]) < 0.1 and abs(vc[2] - p1[2]) < 0.1:
                idx1 = i
            if abs(vc[0] - p2[0]) < 0.1 and abs(vc[1] - p2[1]) < 0.1 and abs(vc[2] - p2[2]) < 0.1:
                idx2 = i
        
        if idx1 >= 0 and idx2 >= 0:
            neo4j_edges.append({
                'from_label': neo4j_nodes[idx1]['label'],
                'to_label': neo4j_nodes[idx2]['label'],
                'relationship': 'CONNECTED_TO',
                'category': 'shared_face'
            })

print("Neo4j Edge Data:")
print(json.dumps(neo4j_edges[:3], indent=2))
print(f"... and {len(neo4j_edges) - 3} more edges")

In [ ]:
# Generate Cypher queries for Neo4j import
print("\nSample Cypher Queries for Neo4j:")
print("=" * 60)

# Create nodes
print("\n-- Create Nodes --")
for node in neo4j_nodes[:2]:
    cypher = f"CREATE (n:Space {{label: '{node['label']}', name: '{node['name']}', volume: {node.get('volume', 0):.1f}, x: {node['x']:.1f}, y: {node['y']:.1f}, z: {node['z']:.1f}}})"
    print(cypher)
print("...")

# Create relationships
print("\n-- Create Relationships --")
for edge in neo4j_edges[:2]:
    cypher = f"MATCH (a:Space {{label: '{edge['from_label']}'}}), (b:Space {{label: '{edge['to_label']}'}}) CREATE (a)-[:CONNECTED_TO]->(b)"
    print(cypher)
print("...")

In [ ]:
# NOTE: Neo4j.ByGraph is not yet implemented in topologic_fast
# The following would be the call in topologicpy:
#
# from topologicpy.Neo4j import Neo4j
# from getpass import getpass
#
# url = "bolt://localhost:7687"
# username = "neo4j"
# password = getpass("Enter Neo4j password: ")
#
# n_graph = Neo4j.ByParameters(url, username, password)
# n_graph = Neo4j.Reset(n_graph)  # Clear existing data
# n_graph = Neo4j.ByGraph(
#     n_graph,
#     house_graph,
#     vertexLabelKey="label",
#     defaultVertexLabel="Untitled",
#     vertexCategoryKey="type",
#     edgeLabelKey="relationship",
#     defaultEdgeLabel="CONNECTED_TO"
# )

print("Neo4j Export Note:")
print("  Neo4j.ByGraph is not yet implemented in topologic_fast.")
print("  ")
print("  To export to Neo4j, you can use the neo4j-python-driver directly:")
print("  ")
print("    from neo4j import GraphDatabase")
print("    driver = GraphDatabase.driver('bolt://localhost:7687', auth=('neo4j', 'password'))")
print("    with driver.session() as session:")
print("        # Execute Cypher queries with the data prepared above")
print("        session.run(cypher_query)")

## Adjacency Matrix Visualization

In [ ]:
# Get adjacency matrix
adj_matrix = house_graph.AdjacencyMatrix()

# Create short labels
short_labels = []
for node in neo4j_nodes:
    name = node['name']
    if 'Living' in name:
        short_labels.append(name.replace('Living Room ', 'LR-'))
    elif 'Kitchen' in name:
        short_labels.append(name.replace('Kitchen ', 'K-'))
    elif 'Bedroom' in name:
        short_labels.append(name.replace('Bedroom ', 'BR-'))
    elif 'Bathroom' in name:
        short_labels.append(name.replace('Bathroom ', 'BT-'))
    elif 'Corridor' in name:
        short_labels.append(name.replace('Corridor ', 'C-'))
    elif 'Home Office' in name:
        short_labels.append(name.replace('Home Office ', 'HO-'))
    elif 'Stairwell' in name:
        short_labels.append('SW')
    else:
        short_labels.append(name[:6])

# Create heatmap
fig = go.Figure(data=go.Heatmap(
    z=adj_matrix,
    x=short_labels,
    y=short_labels,
    colorscale='Blues',
    text=adj_matrix,
    texttemplate='%{text}',
    textfont=dict(size=10),
    colorbar=dict(title='Connected')
))

fig.update_layout(
    title='Room Adjacency Matrix',
    xaxis=dict(title='', tickangle=45, tickfont=dict(size=9)),
    yaxis=dict(title='', autorange='reversed', tickfont=dict(size=9)),
    width=800,
    height=700
)

fig.show()

## Pathfinding

In [ ]:
# Find shortest paths between rooms
print("Shortest Paths:")
print("=" * 60)

# Test a few paths
test_pairs = [
    (0, 5),   # Living Room GF to Home Office GF
    (0, 7),   # Living Room GF to Living Room 1F (via stairwell)
    (3, 12),  # Bedroom GF to Stairwell
]

for start_idx, end_idx in test_pairs:
    if start_idx < len(graph_vertices) and end_idx < len(graph_vertices):
        start_v = graph_vertices[start_idx]
        end_v = graph_vertices[end_idx]
        
        distance = house_graph.Distance(start_v, end_v)
        path = house_graph.Path(start_v, end_v)
        
        start_name = neo4j_nodes[start_idx]['name'] if start_idx < len(neo4j_nodes) else f'Node {start_idx}'
        end_name = neo4j_nodes[end_idx]['name'] if end_idx < len(neo4j_nodes) else f'Node {end_idx}'
        
        print(f"\n{start_name} -> {end_name}:")
        if distance is not None:
            print(f"  Distance: {distance} steps")
            if path:
                path_verts = path.Vertices()
                path_names = []
                for pv in path_verts:
                    pc = pv.Coordinates()
                    for j, gv in enumerate(graph_vertices):
                        gc = gv.Coordinates()
                        if abs(pc[0]-gc[0]) < 0.1 and abs(pc[1]-gc[1]) < 0.1 and abs(pc[2]-gc[2]) < 0.1:
                            if j < len(neo4j_nodes):
                                path_names.append(neo4j_nodes[j]['name'])
                            break
                print(f"  Path: {' -> '.join(path_names)}")
        else:
            print(f"  No path found")

## Summary

This notebook demonstrated:

1. **Building Creation**: Creating a two-storey house with multiple rooms
2. **CellComplex**: Combining rooms into a topological structure
3. **Dual Graph**: Generating a room connectivity graph
4. **Graph Analysis**: Analyzing connectivity, degrees, and paths
5. **Neo4j Preparation**: Structuring data for Neo4j import
6. **Visualization**: 3D views and adjacency matrix heatmaps

### Not Yet Implemented in topologic_fast

The following topologicpy features are not yet available:

- `Neo4j.ByParameters()` - Neo4j connection
- `Neo4j.ByGraph()` - Direct graph export to Neo4j
- `Neo4j.Reset()` - Reset Neo4j database
- `Topology.AddApertures()` - Add doors/windows as apertures
- `Topology.AddContent()` - Add furniture as cell contents
- `Dictionary` integration with topology objects
- `CellComplex.Decompose()` - Decompose into component faces
- `Plotly.DataByGraph()` - Native Plotly graph visualization

For Neo4j export, use the neo4j-python-driver with the data prepared in this notebook.